# CS:GO RAG – asystent

Notebook wczytuje pliki `.md` z folderu `data/`, buduje baze wektorowa Chroma i odpowiada na pytania przez lokalny model **llama3** (Ollama).

**Wymagania:** uruchomiony `ollama serve` oraz modele `llama3` i `nomic-embed-text`.

**Jak uzywac:** uruchamiaj komórki po kolei od góry (Run All lub Shift+Enter). Kernel: `csgorag`.

In [ ]:
import os
from pathlib import Path

# Ustaw True, zeby przebudowac baze od zera po zmianie plikow w data/
REBUILD = False

def find_project_dir():
    cwd = Path.cwd()
    if (cwd / "data").is_dir():
        return cwd
    for parent in cwd.parents:
        if (parent / "data").is_dir():
            return parent
    return cwd

PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "data"
CHROMA_DIR = PROJECT_DIR / "chroma_db"

os.chdir(PROJECT_DIR)
print("Folder projektu:", PROJECT_DIR)
print("Pliki w data/:", sorted(p.name for p in DATA_DIR.glob("*.md")))

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Importy OK")

In [ ]:
loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(docs)

print(f"Wczytano {len(docs)} dokumentow, podzielono na {len(chunks)} fragmentow")

In [ ]:
import shutil

embeddings = OllamaEmbeddings(model="nomic-embed-text")

if REBUILD and CHROMA_DIR.is_dir():
    shutil.rmtree(CHROMA_DIR)
    print("Usunieto stara baze chroma_db.")

if not REBUILD and CHROMA_DIR.is_dir() and any(CHROMA_DIR.iterdir()):
    vectorstore = Chroma(persist_directory=str(CHROMA_DIR), embedding_function=embeddings)
    print("Zaladowano istniejaca baze chroma_db.")
else:
    vectorstore = Chroma.from_documents(
        chunks,
        embeddings,
        persist_directory=str(CHROMA_DIR),
    )
    print("Zbudowano nowa baze chroma_db.")

In [ ]:
llm = ChatOllama(model="llama3", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


def zapytaj(pytanie: str) -> None:
    """Zadaj pytanie asystentowi i wyswietl odpowiedz ze zrodlami."""
    dokumenty = retriever.invoke(pytanie)
    kontekst = "\n\n".join(d.page_content for d in dokumenty)
    prompt = (
        "Odpowiedz na pytanie wylacznie na podstawie ponizszego kontekstu. "
        "Jesli czegos nie ma w kontekscie, napisz, ze nie wiesz.\n\n"
        f"KONTEKST:\n{kontekst}\n\nPYTANIE: {pytanie}"
    )
    odpowiedz = llm.invoke(prompt).content

    print("PYTANIE:", pytanie)
    print("\n" + odpowiedz)
    zrodla = sorted(set(d.metadata.get("source") for d in dokumenty))
    print("\nZrodla:", zrodla)


print("Asystent gotowy. Uzyj zapytaj('twoje pytanie') lub komórek ponizej.")

## Przykladowe pytania

Uruchom dowolna komórkę ponizej albo wpisz wlasne pytanie w ostatniej komórce.

In [ ]:
zapytaj("Ile kosztuje AK-47 i jakie ma obrazenia?")

In [ ]:
zapytaj("Wymien callouty na mapie Mirage.")

In [ ]:
zapytaj("Co to jest loss bonus i jak rosnie?")

## Tryb interaktywny

Uruchom komórkę ponizej i wpisuj pytania w oknie dialogowym. Wpisz `koniec`, zeby zakonczyc.

In [ ]:
print("Zadawaj pytania o CS:GO. Wpisz 'koniec', zeby zakonczyc.\n")

while True:
    pytanie = input("Twoje pytanie: ").strip()
    if pytanie.lower() in ("koniec", "exit", "quit", ""):
        print("Zakonczono.")
        break
    print()
    zapytaj(pytanie)
    print("\n" + "-" * 60 + "\n")

## Wlasne pytanie

Edytuj tekst ponizej i uruchom komórkę.

In [ ]:
MOJE_PYTANIE = "Jakie launch options sa przydatne w CS:GO?"

zapytaj(MOJE_PYTANIE)